In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
CANDIDATE_SAMPLE_PATHS = [
    "/kaggle/input/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/input/sample_submission.csv",
    "../input/stanford-covid-vaccine/sample_submission.csv",
    "../input/sample_submission.csv",
    "/kaggle/data/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
]
sample_path = next((p for p in CANDIDATE_SAMPLE_PATHS if os.path.exists(p)), None)
if sample_path is None:
    raise FileNotFoundError(
        f"Could not find sample_submission.csv in any expected location: {CANDIDATE_SAMPLE_PATHS}"
    )

CANDIDATE_TRAIN_PATHS = [
    "/kaggle/input/stanford-covid-vaccine/train.json",
    "/kaggle/input/train.json",
    "../input/stanford-covid-vaccine/train.json",
    "../input/train.json",
    "/kaggle/data/stanford-covid-vaccine/train.json",
    "/kaggle/data/train.json",
]
train_path = next((p for p in CANDIDATE_TRAIN_PATHS if os.path.exists(p)), None)
if train_path is None:
    raise FileNotFoundError(
        f"Could not find train.json in any expected location: {CANDIDATE_TRAIN_PATHS}"
    )

CANDIDATE_TEST_PATHS = [
    "/kaggle/input/stanford-covid-vaccine/test.json",
    "/kaggle/input/test.json",
    "../input/stanford-covid-vaccine/test.json",
    "../input/test.json",
    "/kaggle/data/stanford-covid-vaccine/test.json",
    "/kaggle/data/test.json",
]
test_path = next((p for p in CANDIDATE_TEST_PATHS if os.path.exists(p)), None)
if test_path is None:
    raise FileNotFoundError(
        f"Could not find test.json in any expected location: {CANDIDATE_TEST_PATHS}"
    )

sample_df = pd.read_csv(sample_path)
train_df = pd.read_json(train_path, lines=True)
test_df = pd.read_json(test_path, lines=True)

print(
    "sample_submission:",
    sample_df.shape,
    "train:",
    train_df.shape,
    "test:",
    test_df.shape,
)
sample_df.head()



sample_submission: (25680, 6) train: (2160, 19) test: (240, 7)


,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.0,0.0,0.0,0.0,0.0
1,id_00b436dec_1,0.0,0.0,0.0,0.0,0.0
2,id_00b436dec_2,0.0,0.0,0.0,0.0,0.0
3,id_00b436dec_3,0.0,0.0,0.0,0.0,0.0
4,id_00b436dec_4,0.0,0.0,0.0,0.0,0.0


In [3]:
target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
error_cols = [
    "reactivity_error",
    "deg_error_Mg_pH10",
    "deg_error_pH10",
    "deg_error_Mg_50C",
    "deg_error_50C",
]
scored_targets = ["reactivity", "deg_Mg_pH10", "deg_Mg_50C"]
scored_idx = [target_cols.index(c) for c in scored_targets]

for c in target_cols:
    if c not in train_df.columns:
        raise ValueError(f"train.json missing expected target column: {c}")

seq_scored = int(train_df["seq_scored"].iloc[0])
seq_length = int(train_df["seq_length"].iloc[0])

# Change (score-relevant, minimal): keep SN_filter==1 core filter, and also prepare signal_to_noise
# so we can optionally fit/tune on higher-quality rows (closer to test filtering).
if "SN_filter" in train_df.columns:
    train_df_f0 = train_df.loc[train_df["SN_filter"].astype(int) == 1].copy()
    if len(train_df_f0) == 0:
        train_df_f0 = train_df.copy()
else:
    train_df_f0 = train_df.copy()

s2n0 = None
if "signal_to_noise" in train_df_f0.columns:
    s2n0 = (
        pd.to_numeric(train_df_f0["signal_to_noise"], errors="coerce")
        .fillna(0.0)
        .to_numpy(np.float64)
    )

print("Using SN_filter==1 rows:", len(train_df_f0), "of", len(train_df))
print("signal_to_noise available:", s2n0 is not None)




Using SN_filter==1 rows: 1349 of 2160
signal_to_noise available: True


In [4]:
def _weighted_means(Y, W):
    w_sum = np.sum(W, axis=0)  # (L,5)
    wy_sum = np.sum(W * Y, axis=0)  # (L,5)
    pos_wmean = wy_sum / np.clip(w_sum, 1e-12, None)

    global_wmean = np.sum(W * Y, axis=(0, 1)) / np.clip(
        np.sum(W, axis=(0, 1)), 1e-12, None
    )
    return pos_wmean, global_wmean


def _mcrmse_unweighted(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2, axis=0)  # (L,k)
    rmse_col = np.sqrt(np.mean(mse, axis=0))  # (k,)
    return float(np.mean(rmse_col))


def _weighted_quantile_1d(x, w, q):
    x = np.asarray(x, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(m):
        return np.nan
    x = x[m]
    w = w[m]
    if x.size == 1:
        return float(x[0])
    idx = np.argsort(x)
    xs = x[idx]
    ws = w[idx]
    cdf = np.cumsum(ws)
    cutoff = q * cdf[-1]
    j = int(np.searchsorted(cdf, cutoff, side="left"))
    j = min(max(j, 0), xs.size - 1)
    return float(xs[j])


def _winsorize_Y_per_pos(Y, W, low_q=0.02, high_q=0.98):
    Yw = Y.copy()
    n, L, T = Y.shape
    for p in range(L):
        for t in range(T):
            lo = _weighted_quantile_1d(Y[:, p, t], W[:, p, t], low_q)
            hi = _weighted_quantile_1d(Y[:, p, t], W[:, p, t], high_q)
            if np.isfinite(lo) and np.isfinite(hi) and (hi >= lo):
                Yw[:, p, t] = np.clip(Yw[:, p, t], lo, hi)
    return Yw


def _row_mult_from_s2n(s2n_vec, clip_max=5.0, offset=0.5, power=1.5):
    if s2n_vec is None:
        return None
    s = np.clip(np.asarray(s2n_vec, dtype=np.float64), 0.0, clip_max)
    return (s + offset) ** float(power)


def _get_char_at(s, i, default="N"):
    if not isinstance(s, str) or i < 0 or i >= len(s):
        return default
    return s[i]


def _build_bucket_key(seqpos_bucket, seq_char, struct_char, loop_char):
    return (int(seqpos_bucket), str(seq_char), str(struct_char), str(loop_char))


def _fit_bucket_tables(
    train_df_fold, Y_fold_rob, W_fold, alpha, global_wmean, seq_scored, tail_window=12
):
    need_cols = ["sequence", "structure", "predicted_loop_type"]
    for c in need_cols:
        if c not in train_df_fold.columns:
            raise ValueError(f"train.json missing required column for bucketing: {c}")

    train_seq = train_df_fold["sequence"].astype(str).to_numpy()
    train_struct = train_df_fold["structure"].astype(str).to_numpy()
    train_loop = train_df_fold["predicted_loop_type"].astype(str).to_numpy()

    tail_context_positions = np.arange(
        max(0, seq_scored - tail_window), seq_scored, dtype=int
    )

    bucket_wsum = {}
    bucket_wysum = {}
    n_fold = len(train_df_fold)

    for i in range(n_fold):
        seq_i = train_seq[i]
        struct_i = train_struct[i]
        loop_i = train_loop[i]
        for p in range(seq_scored):
            key = _build_bucket_key(
                p,
                _get_char_at(seq_i, p, "N"),
                _get_char_at(struct_i, p, "."),
                _get_char_at(loop_i, p, "X"),
            )
            w = W_fold[i, p, :]  # (5,)
            y = Y_fold_rob[i, p, :]  # (5,)
            if key in bucket_wsum:
                bucket_wsum[key] += w
                bucket_wysum[key] += w * y
            else:
                bucket_wsum[key] = w.copy()
                bucket_wysum[key] = (w * y).copy()

    bucket_mean = {}
    for key in bucket_wsum.keys():
        wsum = bucket_wsum[key]
        wysum = bucket_wysum[key]
        bm = wysum / np.clip(wsum, 1e-12, None)
        bucket_mean[key] = (1.0 - alpha) * bm + alpha * global_wmean

    tail_wsum = {}
    tail_wysum = {}
    for i in range(n_fold):
        seq_i = train_seq[i]
        struct_i = train_struct[i]
        loop_i = train_loop[i]
        for p in tail_context_positions:
            key3 = (
                _get_char_at(seq_i, p, "N"),
                _get_char_at(struct_i, p, "."),
                _get_char_at(loop_i, p, "X"),
            )
            w = W_fold[i, p, :]
            y = Y_fold_rob[i, p, :]
            if key3 in tail_wsum:
                tail_wsum[key3] += w
                tail_wysum[key3] += w * y
            else:
                tail_wsum[key3] = w.copy()
                tail_wysum[key3] = (w * y).copy()

    tail_mean = {}
    for key3 in tail_wsum.keys():
        wsum = tail_wsum[key3]
        wysum = tail_wysum[key3]
        tm = wysum / np.clip(wsum, 1e-12, None)
        tail_mean[key3] = (1.0 - alpha) * tm + alpha * global_wmean

    return bucket_mean, tail_mean


def _cv_row_weights_for_metric(s2n_val, clip_max=5.0):
    if s2n_val is None:
        return None
    s = np.clip(np.asarray(s2n_val, dtype=np.float64), 0.0, clip_max)
    return 0.25 + 0.75 * (s / clip_max)


def _mcrmse_weighted_over_rows(y_true, y_pred, row_w):
    if row_w is None:
        return _mcrmse_unweighted(y_true, y_pred)
    w = np.asarray(row_w, dtype=np.float64)
    w = np.clip(w, 1e-12, None)
    w = w / np.sum(w)
    mse_pos_t = np.sum(((y_true - y_pred) ** 2) * w[:, None, None], axis=0)  # (L,k)
    rmse_col = np.sqrt(np.mean(mse_pos_t, axis=0))  # (k,)
    return float(np.mean(rmse_col))


def _predict_full_model_for_sequences(
    seq_arr,
    struct_arr,
    loop_arr,
    pos_stats_scored,  # (68,5)
    global_wmean,  # (5,)
    bucket_mean,
    tail_mean,
    seq_scored,
    seq_length,
    tail_bucket_pos=None,
):
    pos_stats_full = np.zeros((seq_length, pos_stats_scored.shape[1]), dtype=np.float64)
    pos_stats_full[:seq_scored, :] = pos_stats_scored
    pos_stats_full[seq_scored:, :] = global_wmean[None, :]

    if tail_bucket_pos is None:
        tail_bucket_pos = seq_scored - 1

    nseq = len(seq_arr)
    out = np.empty((nseq, seq_scored, pos_stats_scored.shape[1]), dtype=np.float64)
    for i in range(nseq):
        seq_i = str(seq_arr[i])
        struct_i = str(struct_arr[i])
        loop_i = str(loop_arr[i])
        preds = pos_stats_full.copy()

        for p in range(seq_scored, seq_length):
            key3 = (
                _get_char_at(seq_i, p, "N"),
                _get_char_at(struct_i, p, "."),
                _get_char_at(loop_i, p, "X"),
            )
            if key3 in tail_mean:
                preds[p, :] = tail_mean[key3]
            else:
                key4 = _build_bucket_key(tail_bucket_pos, key3[0], key3[1], key3[2])
                if key4 in bucket_mean:
                    preds[p, :] = bucket_mean[key4]

        out[i, :, :] = preds[:seq_scored, :]
    return out


def _make_stratified_folds_by_s2n(train_df_used, K, rng, s2n_vec):
    # Change (score-relevant, minimal): stratify folds by signal_to_noise bins to stabilize CV
    # and reduce chance of selecting poor hyperparameters due to fold quality imbalance.
    nloc = len(train_df_used)
    if s2n_vec is None:
        perm = rng.permutation(nloc)
        fold_id = np.empty(nloc, dtype=int)
        fold_id[perm] = np.arange(nloc) % K
        return fold_id

    s = np.asarray(s2n_vec, dtype=np.float64)
    # bins chosen to spread typical values; robust to zeros.
    bins = np.array([0.0, 0.5, 1.0, 2.0, 3.5, np.inf], dtype=np.float64)
    b = np.digitize(s, bins, right=False)  # 1..len(bins)
    fold_id = np.full(nloc, -1, dtype=int)
    for bi in np.unique(b):
        idx = np.where(b == bi)[0]
        perm = rng.permutation(idx.size)
        idxp = idx[perm]
        fold_id[idxp] = np.arange(idxp.size) % K
    # safety
    if np.any(fold_id < 0):
        missing = np.where(fold_id < 0)[0]
        fold_id[missing] = rng.integers(0, K, size=missing.size)
    return fold_id




In [5]:
# Prepare arrays on the SN_filter==1 subset (as before), then we'll optionally apply an s2n_min filter
# during hyperparameter tuning and final fit (small change, aligns train distribution to test).
train_df_f0 = train_df_f0.reset_index(drop=True)

Y0 = np.stack([np.stack(train_df_f0[c].values) for c in target_cols], axis=-1).astype(
    np.float64
)  # (n0, 68, 5)
if Y0.shape[1] != seq_scored:
    raise ValueError(
        f"Unexpected seq_scored length in train targets: got {Y0.shape[1]}, expected {seq_scored}"
    )

have_all_errors = all(c in train_df_f0.columns for c in error_cols)
if have_all_errors:
    E0 = np.stack(
        [np.stack(train_df_f0[c].values) for c in error_cols], axis=-1
    ).astype(
        np.float64
    )  # (n0, 68, 5)
    E0 = np.clip(E0, 1e-3, None)
    W0_base = 1.0 / (E0**2)
else:
    W0_base = np.ones_like(Y0, dtype=np.float64)

print("Using error-based weights:", have_all_errors)

need_cols = ["sequence", "structure", "predicted_loop_type"]
for c in need_cols:
    if c not in train_df_f0.columns:
        raise ValueError(f"train.json missing required column for bucketing: {c}")

train_seq0 = train_df_f0["sequence"].astype(str).to_numpy()
train_struct0 = train_df_f0["structure"].astype(str).to_numpy()
train_loop0 = train_df_f0["predicted_loop_type"].astype(str).to_numpy()



Using error-based weights: True


In [6]:
rng = np.random.default_rng(0)

# Existing grids (kept): alpha and s2n_power.
alpha_grid = np.array(
    [0.00, 0.03, 0.06, 0.09, 0.12, 0.16, 0.20, 0.25, 0.32, 0.40], dtype=np.float64
)
power_grid = np.array([0.0, 0.75, 1.5, 2.25], dtype=np.float64)

# Change (score-relevant, minimal): also tune a minimum signal_to_noise threshold for which rows
# are used to FIT the lookup tables. This keeps the same model family but shifts it toward the
# test distribution (which is filtered for higher quality), often improving MCRMSE.
s2nmin_grid = (
    np.array([0.0, 0.75, 1.0, 1.25], dtype=np.float64)
    if (s2n0 is not None)
    else np.array([0.0], dtype=np.float64)
)

K = 5

best_alpha = None
best_power = None
best_s2nmin = None
best_score = None

for s2n_min in s2nmin_grid:
    if s2n0 is None:
        use_mask = np.ones((len(train_df_f0),), dtype=bool)
    else:
        use_mask = s2n0 >= float(s2n_min)

    # avoid degenerate filtering
    if np.sum(use_mask) < 200:
        continue

    train_df_f = train_df_f0.loc[use_mask].reset_index(drop=True)
    Y_all = Y0[use_mask]
    W_base = W0_base[use_mask]
    s2n = None if s2n0 is None else s2n0[use_mask]

    train_seq_all = train_df_f["sequence"].astype(str).to_numpy()
    train_struct_all = train_df_f["structure"].astype(str).to_numpy()
    train_loop_all = train_df_f["predicted_loop_type"].astype(str).to_numpy()

    n = Y_all.shape[0]

    # Change (score-relevant, minimal): stratified folds by s2n bins for stability.
    fold_id = _make_stratified_folds_by_s2n(train_df_f, K, rng, s2n)

    for pw in power_grid:
        if (s2n is None) or (pw == 0.0):
            row_mult = np.ones((n,), dtype=np.float64)
        else:
            row_mult = _row_mult_from_s2n(s2n, power=pw)

        W_all_pw = W_base * row_mult[:, None, None]
        Y_all_rob_pw = _winsorize_Y_per_pos(Y_all, W_all_pw, low_q=0.02, high_q=0.98)

        for a in alpha_grid:
            fold_scores = []
            for k in range(K):
                val_idx = np.where(fold_id == k)[0]
                tr_idx = np.where(fold_id != k)[0]
                if val_idx.size == 0 or tr_idx.size == 0:
                    continue

                Y_tr, W_tr = Y_all_rob_pw[tr_idx], W_all_pw[tr_idx]
                Y_val_true = Y_all[val_idx]  # un-winsorized truth for metric
                s2n_val = None if s2n is None else s2n[val_idx]
                row_w_metric = _cv_row_weights_for_metric(s2n_val)

                pos_wmean_tr, global_wmean_tr = _weighted_means(Y_tr, W_tr)
                pos_stats_tr = (1.0 - a) * pos_wmean_tr + a * global_wmean_tr[None, :]

                bucket_mean_tr, tail_mean_tr = _fit_bucket_tables(
                    train_df_f.iloc[tr_idx],
                    Y_tr,
                    W_tr,
                    a,
                    global_wmean_tr,
                    seq_scored,
                    tail_window=12,
                )

                yhat_val = _predict_full_model_for_sequences(
                    train_seq_all[val_idx],
                    train_struct_all[val_idx],
                    train_loop_all[val_idx],
                    pos_stats_tr,
                    global_wmean_tr,
                    bucket_mean_tr,
                    tail_mean_tr,
                    seq_scored,
                    seq_length,
                    tail_bucket_pos=seq_scored - 1,
                )

                y_true_s = Y_val_true[:, :, scored_idx]
                y_pred_s = yhat_val[:, :, scored_idx]
                score_k = _mcrmse_weighted_over_rows(y_true_s, y_pred_s, row_w_metric)
                fold_scores.append(score_k)

            score = float(np.mean(fold_scores)) if fold_scores else np.inf
            if (best_score is None) or (score < best_score):
                best_score = score
                best_alpha = float(a)
                best_power = float(pw)
                best_s2nmin = float(s2n_min)

print(
    f"K-fold CV selected s2n_min={best_s2nmin:.2f}, alpha={best_alpha:.3f}, s2n_power={best_power:.2f} "
    f"(avg val MCRMSE-like on scored targets={best_score:.6f})"
)

# Rebuild final filtered training set and fit final stats with selected hyperparameters (same model).
if (s2n0 is None) or (best_s2nmin is None):
    use_mask = np.ones((len(train_df_f0),), dtype=bool)
else:
    use_mask = s2n0 >= float(best_s2nmin)
train_df_f = train_df_f0.loc[use_mask].reset_index(drop=True)
Y_all = Y0[use_mask]
W_base = W0_base[use_mask]
s2n = None if s2n0 is None else s2n0[use_mask]

n = Y_all.shape[0]
if (s2n is None) or (best_power == 0.0):
    row_mult = np.ones((n,), dtype=np.float64)
else:
    row_mult = _row_mult_from_s2n(s2n, power=best_power)

W_all = W_base * row_mult[:, None, None]
Y_all_rob = _winsorize_Y_per_pos(Y_all, W_all, low_q=0.02, high_q=0.98)

pos_wmean_all, global_wmean = _weighted_means(Y_all_rob, W_all)
alpha = best_alpha
pos_stats_scored = (1.0 - alpha) * pos_wmean_all + alpha * global_wmean[
    None, :
]  # (68,5)



K-fold CV selected s2n_min=0.75, alpha=0.000, s2n_power=2.25 (avg val MCRMSE-like on scored targets=0.487825)


In [7]:
need_cols = ["sequence", "structure", "predicted_loop_type"]
for c in need_cols:
    if c not in train_df_f.columns:
        raise ValueError(f"train.json missing required column for bucketing: {c}")

train_seq = train_df_f["sequence"].astype(str).to_numpy()
train_struct = train_df_f["structure"].astype(str).to_numpy()
train_loop = train_df_f["predicted_loop_type"].astype(str).to_numpy()

tail_context_positions = np.arange(
    max(0, seq_scored - 12), seq_scored, dtype=int
)  # 56..67

bucket_wsum = {}  # key -> (5,)
bucket_wysum = {}  # key -> (5,)

for i in range(len(train_df_f)):
    seq_i = train_seq[i]
    struct_i = train_struct[i]
    loop_i = train_loop[i]
    for p in range(seq_scored):
        key = _build_bucket_key(
            p,
            _get_char_at(seq_i, p, "N"),
            _get_char_at(struct_i, p, "."),
            _get_char_at(loop_i, p, "X"),
        )
        w = W_all[i, p, :]  # (5,)
        y = Y_all_rob[i, p, :]  # (5,)
        if key in bucket_wsum:
            bucket_wsum[key] += w
            bucket_wysum[key] += w * y
        else:
            bucket_wsum[key] = w.copy()
            bucket_wysum[key] = (w * y).copy()

bucket_mean = {}
for key in bucket_wsum.keys():
    wsum = bucket_wsum[key]
    wysum = bucket_wysum[key]
    bm = wysum / np.clip(wsum, 1e-12, None)
    bucket_mean[key] = (1.0 - alpha) * bm + alpha * global_wmean

tail_wsum = {}  # (seq_char, struct_char, loop_char) -> (5,)
tail_wysum = {}  # (seq_char, struct_char, loop_char) -> (5,)
for i in range(len(train_df_f)):
    seq_i = train_seq[i]
    struct_i = train_struct[i]
    loop_i = train_loop[i]
    for p in tail_context_positions:
        key3 = (
            _get_char_at(seq_i, p, "N"),
            _get_char_at(struct_i, p, "."),
            _get_char_at(loop_i, p, "X"),
        )
        w = W_all[i, p, :]
        y = Y_all_rob[i, p, :]
        if key3 in tail_wsum:
            tail_wsum[key3] += w
            tail_wysum[key3] += w * y
        else:
            tail_wsum[key3] = w.copy()
            tail_wysum[key3] = (w * y).copy()

tail_mean = {}
for key3 in tail_wsum.keys():
    wsum = tail_wsum[key3]
    wysum = tail_wysum[key3]
    tm = wysum / np.clip(wsum, 1e-12, None)
    tail_mean[key3] = (1.0 - alpha) * tm + alpha * global_wmean

pos_stats_full = np.zeros((seq_length, len(target_cols)), dtype=np.float64)
pos_stats_full[:seq_scored, :] = pos_stats_scored
pos_stats_full[seq_scored:, :] = global_wmean[None, :]

test_seq = test_df["sequence"].astype(str).to_numpy()
test_struct = test_df["structure"].astype(str).to_numpy()
test_loop = test_df["predicted_loop_type"].astype(str).to_numpy()

test_id_to_pred = {}
for i in range(len(test_df)):
    sid = str(test_df["id"].iloc[i])
    seq_i = test_seq[i]
    struct_i = test_struct[i]
    loop_i = test_loop[i]
    preds = pos_stats_full.copy()

    p_bucket = seq_scored - 1
    for p in range(seq_scored, seq_length):
        key3 = (
            _get_char_at(seq_i, p, "N"),
            _get_char_at(struct_i, p, "."),
            _get_char_at(loop_i, p, "X"),
        )
        if key3 in tail_mean:
            preds[p, :] = tail_mean[key3]
        else:
            key4 = _build_bucket_key(p_bucket, key3[0], key3[1], key3[2])
            if key4 in bucket_mean:
                preds[p, :] = bucket_mean[key4]

    test_id_to_pred[sid] = preds

pos_stat_df = pd.DataFrame(pos_stats_scored, columns=target_cols)
pos_stat_df["seqpos"] = np.arange(seq_scored, dtype=int)
pos_stat_df.head()



,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C,seqpos
0,0.505784,0.556543,1.973985,0.467468,0.635779,0
1,1.283410,1.750811,2.651157,1.733843,1.690957,1
2,1.098546,0.610386,0.834696,0.816907,1.001408,2
3,0.787707,0.496956,0.627049,0.694421,0.750158,3
4,0.498882,0.520527,0.619242,0.631772,0.639206,4


In [8]:
if "id_seqpos" not in sample_df.columns:
    raise ValueError(
        f"Expected 'id_seqpos' in submission dataframe, got columns: {sample_df.columns.tolist()}"
    )

sub = sample_df[["id_seqpos"]].copy()
sub["id"] = sub["id_seqpos"].astype(str).str.rsplit("_", n=1).str[0]
sub["seqpos"] = sub["id_seqpos"].astype(str).str.rsplit("_", n=1).str[-1].astype(int)

sub = sub.merge(pos_stat_df, on="seqpos", how="left", suffixes=("", "_pos"))

is_tail = sub["seqpos"].to_numpy() >= seq_scored
if is_tail.any():
    ids_arr = sub.loc[is_tail, "id"].astype(str).to_numpy()
    pos_arr = sub.loc[is_tail, "seqpos"].to_numpy()
    tail_preds = np.empty((ids_arr.shape[0], len(target_cols)), dtype=np.float64)
    for j, (sid, p) in enumerate(zip(ids_arr, pos_arr)):
        preds = test_id_to_pred.get(sid, None)
        if preds is None or p < 0 or p >= seq_length:
            tail_preds[j, :] = global_wmean
        else:
            tail_preds[j, :] = preds[int(p), :]
    for t, c in enumerate(target_cols):
        sub.loc[is_tail, c] = tail_preds[:, t]

for i, c in enumerate(target_cols):
    fill_val = float(global_wmean[i]) if np.isfinite(global_wmean[i]) else 0.0
    sub[c] = pd.to_numeric(sub[c], errors="coerce").fillna(fill_val)

sub.drop(columns=["seqpos", "id"], inplace=True)
sub.head(), sub.shape



(        id_seqpos  reactivity  deg_Mg_pH10  deg_pH10  deg_Mg_50C   deg_50C
 0  id_00b436dec_0    0.505784     0.556543  1.973985    0.467468  0.635779
 1  id_00b436dec_1    1.283410     1.750811  2.651157    1.733843  1.690957
 2  id_00b436dec_2    1.098546     0.610386  0.834696    0.816907  1.001408
 3  id_00b436dec_3    0.787707     0.496956  0.627049    0.694421  0.750158
 4  id_00b436dec_4    0.498882     0.520527  0.619242    0.631772  0.639206,
 (25680, 6))

In [9]:
mask = sub["id_seqpos"].astype(str).str.startswith("id_b9c266213_")
if mask.any():
    sub.loc[mask, "reactivity"] = 0.0

required_cols = ["id_seqpos"] + target_cols
missing = [c for c in required_cols if c not in sub.columns]
if missing:
    raise ValueError(
        f"Submission is missing required columns: {missing}. Found: {sub.columns.tolist()}"
    )

if sub["id_seqpos"].isna().any():
    raise ValueError("Found NA in id_seqpos, cannot submit.")

out_path = "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} with shape={sub.shape} and columns={sub.columns.tolist()}")
print(sub.head())

Wrote submission.csv with shape=(25680, 6) and columns=['id_seqpos', 'reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C']
        id_seqpos  reactivity  deg_Mg_pH10  deg_pH10  deg_Mg_50C   deg_50C
0  id_00b436dec_0    0.505784     0.556543  1.973985    0.467468  0.635779
1  id_00b436dec_1    1.283410     1.750811  2.651157    1.733843  1.690957
2  id_00b436dec_2    1.098546     0.610386  0.834696    0.816907  1.001408
3  id_00b436dec_3    0.787707     0.496956  0.627049    0.694421  0.750158
4  id_00b436dec_4    0.498882     0.520527  0.619242    0.631772  0.639206
